# 02b — Method 1: CLAP (Contrastive Language-Audio Pre-training)

Use CLAP embeddings to match text prompts to songs via audio-text similarity.

**Prerequisites:** Run `02a_data_feature_prep.ipynb` first.

In [20]:
# --- Shared Setup ---
# This notebook depends on the data and features prepared in 02a.
# Run 02a_data_feature_prep.ipynb first, or load saved artifacts.

import sys
sys.path.insert(0, '..')

import os
import json
import time
import warnings
from pathlib import Path
from typing import TypedDict, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.spatial.distance import cosine as cosine_dist
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity

warnings.filterwarnings('ignore', category=FutureWarning)

from atdj.config import PROCESSED_DIR, DATA_DIR

# --- Load saved artifacts from notebook 02a ---
FEATURES_DIR = Path(PROCESSED_DIR) / 'features_exp'
df_merged = pd.read_pickle(FEATURES_DIR / 'df_merged.pkl')
print(f'Loaded df_merged: {df_merged.shape}')

import pickle
with open(FEATURES_DIR / 'feature_catalog.pkl', 'rb') as f:
    FEATURE_CATALOG = pickle.load(f)

ALL_FEATURE_NAMES = []
FEATURE_DESC_MAP = {}
for group, features in FEATURE_CATALOG.items():
    for fname, fdesc in features:
        ALL_FEATURE_NAMES.append(fname)
        FEATURE_DESC_MAP[fname] = fdesc
AVAILABLE_FEATURES = [f for f in ALL_FEATURE_NAMES if f in df_merged.columns]

with open(FEATURES_DIR / 'feature_prompt.txt', encoding='utf-8') as f:
    FEATURE_PROMPT = f.read()

print(f'Feature catalog: {len(AVAILABLE_FEATURES)} available features across {len(FEATURE_CATALOG)} groups')

# Additional imports for CLAP
import torch

# Audio file directories — find_audio_files() searches both raw/ and cortinas/
AUDIO_BASE_DIR = Path(DATA_DIR) / 'raw'

Loaded df_merged: (184, 213)
Feature catalog: 99 available features across 15 groups


In [21]:
# --- CLAP Setup ---
try:
    from transformers import ClapModel, ClapProcessor
    import librosa
    CLAP_AVAILABLE = True
    print('CLAP available via transformers')
except ImportError:
    CLAP_AVAILABLE = False
    print('CLAP not available')

CLAP available via transformers


In [22]:
# Standardized output format — all methods return this structure
class SongMatch(TypedDict):
    filename: str
    score: float              # 0-1, higher = better match
    matched_features: dict    # feature_name: actual_value
    explanation: str          # why this song matches

class MatchResult(TypedDict):
    prompt: str
    method: str               # 'clap' | 'cot_llm_a' | 'cot_llm_b' | 'small_model'
    top_k_songs: list         # list of SongMatch
    bottom_k_songs: list      # worst-matching songs for human validation
    feature_ranges_used: dict # feature_name: {min, max, direction} (methods 2 & 3)
    metadata: dict            # method-specific (model name, latency, etc.)


# Test prompts used across all methods
TEST_PROMPTS = [
    'energetic tango with strong rhythm for experienced dancers',
    'melancholic and slow, perfect for a late-night vals',
    'bright and cheerful milonga with clear melody',
    'dramatic tango with heavy bandoneon and dark mood',
    'smooth and relaxed, good for warming up the floor',
    'classic golden-age tango from the 40s, warm and nostalgic',
    'a lively vals from the 50s with a strong orchestra',
    'need a cortina — something upbeat and non-tango to reset the floor',
]

TOP_K = 10
BOTTOM_K = 5  # worst matches for human validation  # number of songs to return per query

# Collect all results here for cross-method comparison
ALL_RESULTS: dict[str, list[MatchResult]] = {}  # method_name -> list of results

print(f'Test prompts: {len(TEST_PROMPTS)}')
print(f'Top-K: {TOP_K}')
for i, p in enumerate(TEST_PROMPTS, 1):
    print(f'  {i}. "{p}"')

Test prompts: 8
Top-K: 10
  1. "energetic tango with strong rhythm for experienced dancers"
  2. "melancholic and slow, perfect for a late-night vals"
  3. "bright and cheerful milonga with clear melody"
  4. "dramatic tango with heavy bandoneon and dark mood"
  5. "smooth and relaxed, good for warming up the floor"
  6. "classic golden-age tango from the 40s, warm and nostalgic"
  7. "a lively vals from the 50s with a strong orchestra"
  8. "need a cortina — something upbeat and non-tango to reset the floor"


In [23]:
def load_clap_model():
    """Load CLAP model and processor from HuggingFace."""
    model = ClapModel.from_pretrained('laion/larger_clap_music_and_speech')
    processor = ClapProcessor.from_pretrained('laion/larger_clap_music_and_speech')
    model.eval()
    return model, processor


def encode_text_clap(prompts: list[str], model, processor) -> np.ndarray:
    """Encode text prompts to CLAP embeddings. Returns (N, 512) array."""
    inputs = processor(text=prompts, return_tensors='pt', padding=True)
    with torch.no_grad():
        text_embeds = model.get_text_features(**inputs)
    out = text_embeds if isinstance(text_embeds, torch.Tensor) else text_embeds.pooler_output
    return out.detach().cpu().numpy()


def encode_audio_clap(audio_path: str | Path, model, processor,
                      target_sr: int = 48000) -> np.ndarray:
    """Encode a single audio file to CLAP embedding. Returns (512,) array."""
    y, sr = librosa.load(str(audio_path), sr=target_sr, mono=True)
    inputs = processor(audio=y, sampling_rate=target_sr, return_tensors='pt')
    with torch.no_grad():
        audio_embed = model.get_audio_features(**inputs)
    out = audio_embed if isinstance(audio_embed, torch.Tensor) else audio_embed.pooler_output
    return out.detach().cpu().numpy().flatten()


def find_audio_files(df: pd.DataFrame, audio_dir: Path) -> dict[str, Path]:
    """Match CSV filenames to audio files in raw/ and cortinas/ directories."""
    found = {}
    # Search both raw/ and cortinas/ under the data dir
    search_dirs = [audio_dir]
    cortina_dir = audio_dir.parent / 'cortinas'
    if cortina_dir.exists():
        search_dirs.append(cortina_dir)
    all_mp3s = {}
    for d in search_dirs:
        if d.exists():
            for p in d.rglob('*.mp3'):
                if not p.name.startswith('._'):
                    all_mp3s[p.name] = p
    for fname in df['filename']:
        if fname in all_mp3s:
            found[fname] = all_mp3s[fname]
    return found


if CLAP_AVAILABLE:
    print('Loading CLAP model (first run downloads ~2GB)...')
    clap_model, clap_processor = load_clap_model()
    print('CLAP model loaded successfully')
else:
    print('Skipping CLAP — transformers/librosa not available')

Loading CLAP model (first run downloads ~2GB)...


Loading weights:   0%|          | 0/555 [00:00<?, ?it/s]

CLAP model loaded successfully


In [24]:
def encode_features_proxy(df: pd.DataFrame, feature_cols: list[str]) -> np.ndarray:
    """Fallback: build normalized feature vectors when audio files are unavailable.
    Returns (N, D) array where D = len(feature_cols)."""
    feat_df = df[feature_cols].copy()
    feat_df = feat_df.fillna(feat_df.mean())
    feat_df = feat_df.dropna(axis=1)
    scaler = StandardScaler()
    return scaler.fit_transform(feat_df)


def rank_songs_clap(prompt: str, df: pd.DataFrame, model, processor,
                    audio_dir: Path | None = None, top_k: int = 10, bottom_k: int = 5,
                    feature_cols: list[str] | None = None,
                    audio_cache: dict | None = None) -> MatchResult:
    """Rank songs by CLAP similarity. Uses audio encoding if files available,
    otherwise falls back to feature proxy."""
    t0 = time.time()

    # Encode text prompt
    text_embed = encode_text_clap([prompt], model, processor)  # (1, 512)

    # Try audio encoding first
    audio_files = find_audio_files(df, audio_dir) if audio_dir else {}
    use_audio = len(audio_files) >= len(df) * 0.5

    if use_audio:
        print(f'  CLAP: Using audio encoding ({len(audio_files)}/{len(df)} files found)')
        audio_embeds = []
        valid_indices = []
        for idx, row in df.iterrows():
            fname = row['filename']
            if fname in audio_files:
                if audio_cache and fname in audio_cache:
                    embed = audio_cache[fname]
                else:
                    embed = encode_audio_clap(audio_files[fname], model, processor)
                audio_embeds.append(embed)
                valid_indices.append(idx)
        audio_embeds = np.array(audio_embeds)  # (M, 512)
        similarities = cosine_similarity(text_embed, audio_embeds).flatten()
        scores = pd.Series(0.0, index=df.index)
        for i, idx in enumerate(valid_indices):
            scores[idx] = float(similarities[i])
    else:
        print(f'  CLAP: Using feature proxy (audio files not available)')
        if feature_cols is None:
            feature_cols = [f for f in AVAILABLE_FEATURES
                          if f in df.columns and df[f].dtype in ['float64', 'float32', 'int64']]
        feat_matrix = encode_features_proxy(df, feature_cols)
        rng = np.random.RandomState(42)
        proj = rng.randn(text_embed.shape[1], feat_matrix.shape[1])
        proj /= np.linalg.norm(proj, axis=0, keepdims=True)
        text_in_feat_space = text_embed @ proj
        similarities = cosine_similarity(text_in_feat_space, feat_matrix).flatten()
        smin, smax = similarities.min(), similarities.max()
        if smax > smin:
            scores = (similarities - smin) / (smax - smin)
        else:
            scores = np.ones_like(similarities) * 0.5
        scores = pd.Series(scores, index=df.index)

    elapsed = time.time() - t0

    top_indices = scores.nlargest(top_k).index
    bottom_indices = scores.nsmallest(bottom_k).index

    def _build_song_list(indices):
        songs = []
        for idx in indices:
            row = df.loc[idx]
            matched = {}
            for f in AVAILABLE_FEATURES:
                if f in row.index and pd.notna(row[f]):
                    val = row[f]
                    try:
                        matched[f] = round(float(val), 4)
                    except (ValueError, TypeError):
                        pass  # skip non-numeric features
            songs.append({
                'filename': row['filename'],
                'score': round(float(scores[idx]), 4),
                'matched_features': matched,
                'explanation': f'CLAP similarity score: {scores[idx]:.4f}',
            })
        return songs

    return {
        'prompt': prompt,
        'method': 'clap',
        'top_k_songs': _build_song_list(top_indices),
        'bottom_k_songs': _build_song_list(bottom_indices),
        'feature_ranges_used': {},
        'metadata': {
            'model': 'laion/larger_clap_music_and_speech',
            'encoding': 'audio' if use_audio else 'feature_proxy',
            'latency_seconds': round(elapsed, 3),
        },
    }


# ── Pre-compute or load CLAP audio embeddings (incremental) ──
# Loads existing cache, encodes only missing songs, then re-saves.
CLAP_EMBED_PATH = FEATURES_DIR / 'clap_audio_embeddings.npz'
CLAP_AUDIO_CACHE = {}

if CLAP_AVAILABLE:
    # Step 1: Load existing cache (if any)
    if CLAP_EMBED_PATH.exists():
        data = np.load(CLAP_EMBED_PATH, allow_pickle=True)
        cached_filenames = list(data['filenames'])
        cached_embeddings = data['embeddings']
        for fname, emb in zip(cached_filenames, cached_embeddings):
            CLAP_AUDIO_CACHE[str(fname)] = emb
        print(f'Loaded {len(CLAP_AUDIO_CACHE)} cached CLAP audio embeddings from disk')
        print(f'  File: {CLAP_EMBED_PATH.name} ({CLAP_EMBED_PATH.stat().st_size / 1024:.0f} KB)')

    # Step 2: Find all audio files on disk (raw/ + cortinas/)
    audio_files = find_audio_files(df_merged, AUDIO_BASE_DIR)
    print(f'\nAudio files on disk: {len(audio_files)}/{len(df_merged)} songs')

    # Step 3: Determine which songs are missing from cache
    missing = {fname: fpath for fname, fpath in audio_files.items()
               if fname not in CLAP_AUDIO_CACHE}

    # Step 4: Encode only missing files
    if missing:
        print(f'Encoding {len(missing)} missing audio files with CLAP...')
        t_start = time.time()
        n_done = 0
        n_skipped = 0

        for j, (fname, fpath) in enumerate(missing.items()):
            try:
                CLAP_AUDIO_CACHE[fname] = encode_audio_clap(fpath, clap_model, clap_processor)
                n_done += 1
            except Exception as e:
                print(f'  SKIP {fname}: {e}')
                n_skipped += 1
            if (j + 1) % 20 == 0:
                elapsed = time.time() - t_start
                rate = (j + 1) / elapsed
                remaining = (len(missing) - j - 1) / rate
                print(f'  {j+1}/{len(missing)} '
                      f'({elapsed:.0f}s elapsed, ~{remaining:.0f}s remaining, '
                      f'{rate:.1f} files/s)')

        total_time = time.time() - t_start
        print(f'\n=== CLAP encoding complete ===')
        print(f'  New: {n_done} files  |  Skipped: {n_skipped} files')
        print(f'  Total time: {total_time:.1f}s '
              f'({total_time / max(n_done, 1):.2f}s per file)')

        # Re-save updated cache
        if CLAP_AUDIO_CACHE:
            filenames = list(CLAP_AUDIO_CACHE.keys())
            embeddings = np.array([CLAP_AUDIO_CACHE[f] for f in filenames])
            np.savez(CLAP_EMBED_PATH,
                     filenames=np.array(filenames, dtype=object),
                     embeddings=embeddings)
            print(f'  Saved to {CLAP_EMBED_PATH.name} '
                  f'(shape={embeddings.shape}, '
                  f'{CLAP_EMBED_PATH.stat().st_size / 1024:.0f} KB)')
    else:
        if CLAP_AUDIO_CACHE:
            print(f'All {len(CLAP_AUDIO_CACHE)} songs already cached — nothing to encode')
        elif len(audio_files) == 0:
            print('No audio files found — will use feature proxy fallback')

if CLAP_AVAILABLE:
    print('\nTesting CLAP on first prompt...')
    test_result = rank_songs_clap(
        TEST_PROMPTS[0], df_merged, clap_model, clap_processor,
        audio_dir=AUDIO_BASE_DIR, top_k=5, audio_cache=CLAP_AUDIO_CACHE
    )
    print(f'  Method: {test_result["metadata"]["encoding"]}')
    print(f'  Latency: {test_result["metadata"]["latency_seconds"]}s')
    print(f'  Top 3:')
    for s in test_result['top_k_songs'][:3]:
        print(f'    {s["score"]:.4f}  {s["filename"]}')

Loaded 172 cached CLAP audio embeddings from disk
  File: clap_audio_embeddings.npz (348 KB)

Audio files on disk: 172/184 songs
All 172 songs already cached — nothing to encode

Testing CLAP on first prompt...
  CLAP: Using audio encoding (172/184 files found)
  Method: audio
  Latency: 0.086s
  Top 3:
    0.3572  17 Mala Junta.mp3
    0.3526  26 Saludos.mp3
    0.3441  22 Llorar Por Una Mujer.mp3


In [25]:
# Run CLAP across all test prompts
if CLAP_AVAILABLE:
    clap_results = []
    for prompt in TEST_PROMPTS:
        print(f'\nPrompt: "{prompt}"')
        result = rank_songs_clap(
            prompt, df_merged, clap_model, clap_processor,
            audio_dir=AUDIO_BASE_DIR, top_k=TOP_K, bottom_k=BOTTOM_K,
            audio_cache=CLAP_AUDIO_CACHE
        )
        clap_results.append(result)
        print(f'  Top 3: {[s["filename"] for s in result["top_k_songs"][:3]]}')
        print(f'  Bottom 3: {[s["filename"] for s in result["bottom_k_songs"][:3]]}')

    ALL_RESULTS['clap'] = clap_results
    print(f'\n--- CLAP complete: {len(clap_results)} prompts processed ---')
else:
    print('CLAP skipped — dependencies not available')
    ALL_RESULTS['clap'] = []


Prompt: "energetic tango with strong rhythm for experienced dancers"
  CLAP: Using audio encoding (172/184 files found)
  Top 3: ['17 Mala Junta.mp3', '26 Saludos.mp3', '22 Llorar Por Una Mujer.mp3']
  Bottom 3: ['07 Tristezas De La Calle Corrientes.mp3', '08 Volver A Vernos.mp3', '16 Otra Noche.mp3']

Prompt: "melancholic and slow, perfect for a late-night vals"
  CLAP: Using audio encoding (172/184 files found)
  Top 3: ['23 Una Fija.mp3', '17 Tigre Viejo.mp3', '22 Buscandote.mp3']
  Bottom 3: ['26 Relíquias Porteñas.mp3', '11 Rosas De Otoño.mp3', '23 No Me Extraña.mp3']

Prompt: "bright and cheerful milonga with clear melody"
  CLAP: Using audio encoding (172/184 files found)
  Top 3: ['07 De Mi Barrio.mp3', '11 Lagrimas Y Sonrisas.mp3', '17 Mala Junta.mp3']
  Bottom 3: ['23 Pobre Colombina.mp3', '08 Volver A Vernos.mp3', '29 San Bentito De Palermo.mp3']

Prompt: "dramatic tango with heavy bandoneon and dark mood"
  CLAP: Using audio encoding (172/184 files found)
  Top 3: ['25

**Mood Matching vs. Genre/Style Matching — Exploration Notes**

Based on the current prompt, an initial review of the output shows that mood matching is largely functional, though imperfect. 

The ranking order appears meaningful: top results align better with the mood described in the prompt than lower-ranked ones, which is consistent with human evaluation based on relative comparison. 

One notable outlier is the dramatic tango selection, which doesn't match the mood well; this may be a data scope issue that warrants further investigation.

Genre/style matching, however, is clearly insufficient with the current approach. A straightforward fix would be to pre-filter by genre/style before passing candidates to CLAP, allowing CLAP to focus purely on semantic/mood-level matching within a genre-constrained pool.

Next steps:

- Investigate data scope for the tango mismatch

- Implement genre pre-filtering as a pre-CLAP stage

**Note on score range:** Unlike Methods 2 and 3 (which produce scores normalized to 0–1), CLAP scores are raw cosine similarities between audio and text embeddings. In practice these cluster in a narrow positive band (e.g. 0.2–0.4) rather than spanning the full 0–1 range. The ranking within a single query is valid, but scores are not directly comparable across methods without additional normalization.

In [26]:
# --- Save CLAP results ---
results_dir = Path(PROCESSED_DIR) / 'features_exp'
results_dir.mkdir(parents=True, exist_ok=True)

if ALL_RESULTS.get('clap'):
    # Convert numpy types for JSON serialization
    import copy as _copy

    def _jsonify(obj):
        if isinstance(obj, (np.integer,)):
            return int(obj)
        if isinstance(obj, (np.floating,)):
            return float(obj)
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        if isinstance(obj, dict):
            return {k: _jsonify(v) for k, v in obj.items()}
        if isinstance(obj, (list, tuple)):
            return [_jsonify(v) for v in obj]
        return obj

    with open(results_dir / 'clap_results.json', 'w', encoding='utf-8') as f:
        json.dump(_jsonify(ALL_RESULTS['clap']), f, indent=2)
    print(f"Saved {len(ALL_RESULTS['clap'])} CLAP results")
else:
    print('No CLAP results to save')

Saved 8 CLAP results


In [27]:
ALL_RESULTS

{'clap': [{'prompt': 'energetic tango with strong rhythm for experienced dancers',
   'method': 'clap',
   'top_k_songs': [{'filename': '17 Mala Junta.mp3',
     'score': 0.3572,
     'matched_features': {'bpm': 120.2891,
      'onset_rate': 3.2953,
      'duration': 187.2196,
      'tempogram_global_mean': 0.2459,
      'tempogram_max_bin_strength': 0.9863,
      'average_loudness': 0.5021,
      'dynamic_complexity': 5.7436,
      'rms_mean': 0.1029,
      'rms_std': 0.069,
      'mood_happy': 0.4106,
      'mood_relaxed': 0.918,
      'mood_party': 0.0616,
      'mood_acoustic': 0.4943,
      'mood_electronic': 0.0397,
      'mirex_mood_rollicking_cheerful_fun_sweet_amiable_good_natured': 0.4586,
      'mirex_mood_literate_poignant_wistful_bittersweet_autumnal_brooding': 0.2683,
      'mirex_mood_humorous_silly_campy_quirky_whimsical_witty_wry': 0.2583,
      'spectral_complexity_mean': 10.6866,
      'spectral_flux_mean': 0.0451,
      'spectral_contrast_1_mean': 20.6755,
      'sp